# Notebook 2 (GPU — ~30s Startup): Serve Bonsai 2 27B + Vision + 128K + API Key Auth + Ngrok / Cloudflare Tunnel + Auto Keep-Alive (Kaggle & Google Colab)

This notebook launches an OpenAI-compatible `llama-server` endpoint for **Ternary-Bonsai-2-27B** on a cloud GPU, protects all `/v1/*` model routes with a mandatory **API Key** (`--api-key`), exposes the server publicly over **Ngrok** *or* a free zero-config **Cloudflare Quick Tunnel** (`*.trycloudflare.com`), and keeps the session alive automatically.

---

## Choosing a Public Tunnel: Ngrok vs. Cloudflare Quick Tunnel

You can control which tunnel is used via `TUNNEL_PROVIDER` in Cell 1 (`"auto"`, `"ngrok"`, or `"cloudflare"`):

| Feature | **Ngrok** (`"ngrok"`) | **Cloudflare Quick Tunnel** (`"cloudflare"`) |
| :--- | :--- | :--- |
| **Account / Secrets Needed?** | Yes — requires `NGROK_AUTHTOKEN` (free Ngrok account). | **No** — zero signup, zero secrets, works immediately out of the box. |
| **Public URL Type** | Supports your **fixed static domain** (`NGROK_DOMAIN`) so you don't have to update your client URL every session. | **Random temporary domain** (`https://<random-words>.trycloudflare.com`) that changes every time the tunnel restarts. |
| **Best Used When** | You want a permanent URL saved in Cursor, Cline, Open WebUI, or mobile apps. | You don't want to configure Ngrok Secrets, or you just want a fast temporary URL with zero setup. |
| **Default Behavior (`"auto"`)** | Automatically selected if `NGROK_AUTHTOKEN` is provided. | Automatically used as a seamless fallback if `NGROK_AUTHTOKEN` is missing. |

---

## Step-by-Step Setup Guide: What You Should Do

### Option A — On Kaggle
1. **Attach Pre-Downloaded Dataset**: Right sidebar → **Add Input** → **Your Datasets** → select the `bonsai_bundle` dataset created in Notebook 1. *(If skipped, Notebook 2 downloads the files automatically, which takes longer on GPU.)*
2. **Enable GPU**: **Session options** → **Accelerator** → **GPU T4 x2** (recommended for 128K context).
3. **Enable Internet**: **Session options** → **Internet** → **ON** (required for tunneling and CUDA runtime packages).
4. **Configure Secrets (Optional for Cloudflare, Required for Ngrok)**: **Add-ons** → **Secrets** → add and check:
   - `API_KEY` *(recommended)* — your private secret key so anyone who discovers your Base URL still cannot use your model without this key. *(If left empty while `REQUIRE_API_KEY = True`, a strong random key is auto-generated at startup.)*
   - `NGROK_AUTHTOKEN` *(optional)* — add this if you want to use **Ngrok**. If omitted (and `TUNNEL_PROVIDER = "auto"`), the notebook automatically launches a free random **Cloudflare Quick Tunnel** instead.
   - `NGROK_DOMAIN` *(optional)* — your static Ngrok domain if using Ngrok.
5. Click **Run All** (Cell 4 `stop_all()` is commented out so it will not stop your server).

### Option B — On Google Colab
1. **Enable GPU**: **Runtime** → **Change runtime type** → **T4 / L4 / A100**. *(Note: On a single 16 GB T4, lower `CONTEXT_SIZE` to `32768` in Cell 1 if you hit CUDA OOM errors.)*
2. **Configure Secrets (Optional)**: Left sidebar → **Key icon (Secrets)** → add `API_KEY` *(recommended)*, plus `NGROK_AUTHTOKEN` and `NGROK_DOMAIN` *(only if using Ngrok)*, then toggle **Notebook access** ON. *(If you skip Ngrok Secrets, Cloudflare Quick Tunnel is used automatically.)*
3. **Reuse Google Drive Bundle (Optional)**: If you ran Notebook 1 on Colab, set `COLAB_USE_GOOGLE_DRIVE = True` in Cell 1 to load files instantly from `/content/drive/MyDrive/bonsai_bundle`.
4. Run Cells **1 → 3** in order.

### How to Connect Your AI Client (Open WebUI, Cursor, Cline, RikkaHub, Python SDK, etc.)
After Cell 3 finishes starting up, copy the printed connection values into your client:
- **Base URL**: `<PUBLIC_URL>/v1` (either `https://<your-domain>.ngrok-free.app/v1` or `https://<random>.trycloudflare.com/v1`)
- **API Key**: Your `API_KEY` (or the auto-generated `sk-bonsai-...` key printed by Cell 3)
- **Model ID**: `Ternary-Bonsai-2-27B-Abliterated-v2-PQ2_0-MTP.gguf`

> **Security Note:** When `REQUIRE_API_KEY = True`, `llama-server` enforces Bearer token authentication (`Authorization: Bearer <API_KEY>` or `x-api-key: <API_KEY>`) on all model/API endpoints (`/v1/chat/completions`, `/v1/models`, etc.) and rejects unauthenticated requests with **HTTP 401 Unauthorized**. Only `/health` remains public for readiness checks. This protects your GPU even on public `trycloudflare.com` URLs.

---

## Configuration Variables & Functions Reference

### 1. Model & Inference Server Variables (Cell 1)
| Variable | Default | What It Does & When to Change It |
| :--- | :--- | :--- |
| `REPO_ID` | `BoldingBuilds/Ternary-Bonsai-2-27B-Abliterated-v2-PQ2_0-MTP-GGUF` | Fallback Hugging Face repo if the model is not found in your attached Dataset or Drive. |
| `PRISM_RELEASE_TAG` | `prism-b10828-a153697` | Fallback GitHub release tag if `llama-server` needs to be downloaded fresh. |
| `USE_MTP` | `True` | `True`: uses the Multi-Token Prediction (MTP) model + speculative drafting (`--spec-type draft-mtp`) for faster generation. Set `False` if image inputs misbehave. |
| `USE_VISION` | `True` | Enables image input via `mmproj-Q8_0.gguf`. Set `False` to save VRAM if you only need text. |
| `IMAGE_MIN_TOKENS` | `1024` | Minimum visual tokens per image (`--image-min-tokens`). Higher values read small text/details more accurately at the cost of extra prompt tokens. |
| `CONTEXT_SIZE` | `131072` | Total context window (128K tokens). **Lower to `32768` or `65536` on a single 16 GB T4 GPU** if you hit CUDA OOM errors. |
| `KV_CACHE_TYPE` | `"q8_0"` | Quantizes K and V caches to 8-bit (`--cache-type-k/v`) to dramatically reduce VRAM usage at long context lengths. |
| `DEFAULT_THINKING` | `False` | `False`: disables chain-of-thought thinking by default (`--reasoning-budget 0`). `True`: enables reasoning using `REASONING_EFFORT`. |
| `REASONING_EFFORT` | `"medium"` | Passed to the chat template when `DEFAULT_THINKING = True` (`"low"`, `"medium"`, or `"high"`). |
| `SERVER_PORT` | `8081` | Local TCP port for `llama-server` (auto-increments if already in use). |
| `COLAB_USE_GOOGLE_DRIVE` | `False` | Colab-only: set `True` to mount Drive and load/save the bundle at `COLAB_DRIVE_BUNDLE_DIR`. |
| `COLAB_DRIVE_BUNDLE_DIR` | `/content/drive/MyDrive/bonsai_bundle` | Google Drive folder used when `COLAB_USE_GOOGLE_DRIVE = True`. |

### 2. API Key Security & Public Tunnel Variables (Cell 1)
| Variable | Default | What It Does & When to Change It |
| :--- | :--- | :--- |
| `REQUIRE_API_KEY` | `True` | **Recommended `True`.** Enforces `--api-key` on `llama-server` so nobody who knows your Base URL can call your models without the key. |
| `API_KEY` | `""` | Your secret API key. Best practice: store it in **Kaggle/Colab Secrets** under `API_KEY` and leave this blank. If both Secrets and this variable are blank while `REQUIRE_API_KEY = True`, Cell 3 auto-generates a secure random key (`sk-bonsai-...`). |
| `SHOW_API_KEY_IN_OUTPUT` | `False` | `False`: masks keys loaded from Secrets in cell logs (auto-generated keys are always printed once so you can copy them). Set `True` to display the full key. |
| `TUNNEL_PROVIDER` | `"auto"` | Chooses how to expose your server publicly: `"auto"` (Ngrok if `NGROK_AUTHTOKEN` exists, otherwise free Cloudflare Quick Tunnel), `"ngrok"` (force Ngrok), or `"cloudflare"` (force free random `*.trycloudflare.com` domain). |
| `MANUAL_NGROK_AUTHTOKEN` | `""` | Optional Ngrok token fallback if you did not store `NGROK_AUTHTOKEN` in Secrets. Leave empty to use Cloudflare Quick Tunnel automatically when `TUNNEL_PROVIDER = "auto"`. |
| `MANUAL_NGROK_DOMAIN` | `""` | Optional static Ngrok domain fallback if not stored as `NGROK_DOMAIN` in Secrets (ignored when using Cloudflare). |

### 3. Keep-Alive Variables (Cell 1)
| Variable | Default | What It Does & When to Change It |
| :--- | :--- | :--- |
| `ENABLE_KEEP_ALIVE` | `True` | Keeps Cell 3 running in a monitoring loop to prevent platform idle timeouts. Set `False` to start the server in the background and exit Cell 3 immediately. |
| `KEEP_ALIVE_MIN_HOURS` | `5.0` | Target uptime in hours (e.g., `5.0`, `8.0`, `11.5`). |
| `KEEP_ALIVE_PING_INTERVAL` | `45` | Seconds between health checks and stdout heartbeat updates (recommended `30–60`). |
| `KEEP_ALIVE_GPU_PULSE_SEC` | `180` | Sends a tiny 1-token authenticated inference request every N seconds so Kaggle/Colab does not terminate the session for "GPU idle". |
| `KEEP_ALIVE_AUTO_RESTART` | `True` | Automatically restarts `llama-server` or your active tunnel (`ngrok` / `cloudflared`) if either crashes mid-session. |
| `KEEP_ALIVE_ENABLE_BROWSER_JS` | `True` | Injects browser-side JS to click reconnect/idle popups automatically. |

### 4. Built-in Helper Functions (Cells 2–4)
- **`find_existing_file(filename)`**: Searches attached Kaggle inputs, Colab Drive, and local directories for pre-downloaded binaries/models before falling back to internet downloads.
- **`get_cuda_lib_paths(bin_dir)`** & **`test_binary(bin_path)`**: Locates CUDA 12 shared libraries (`LD_LIBRARY_PATH`) and verifies that `llama-server --list-devices` detects the GPU.
- **`get_secret(name, fallback)`**: Reads secrets securely in priority order: Environment Variable → Kaggle/Colab Secrets → manual fallback variable.
- **`start_llama_server(port, api_key)`**: Launches `llama-server` on `127.0.0.1` with GPU offload (`-ngl 99`), Flash Attention (`-fa on`), KV cache quantization, optional MTP/Vision, and `--api-key` authentication, then polls `/health` until ready.
- **`verify_api_key_protection(port, api_key)`**: Runs a quick self-test confirming that `/v1/models` rejects unauthenticated requests (`401`) and accepts your valid Bearer key (`200`).
- **`start_ngrok_tunnel(port, authtoken, domain)`**: Starts an HTTPS Ngrok tunnel forwarding public traffic to `127.0.0.1:<port>`.
- **`start_cloudflare_tunnel(port)`**: Downloads the standalone `cloudflared` binary (if needed) and launches a zero-config Cloudflare Quick Tunnel returning a random `https://<words>.trycloudflare.com` URL.
- **`start_public_tunnel(provider, port, authtoken, domain)`**: Dispatches to Ngrok or Cloudflare based on `TUNNEL_PROVIDER` (handling `"auto"` fallback cleanly).
- **`is_tunnel_alive(provider, proc)`**: Monitors whichever tunnel provider is active so the Keep-Alive loop can auto-restart it if disconnected.
- **`run_gpu_pulse(port, api_key)`**: Sends a 1-token authenticated chat completion request to keep GPU utilization non-zero.
- **`stop_all()`** *(Cell 4)*: Cleanly shuts down Ngrok, Cloudflare (`cloudflared`), and `llama-server` when you are finished.


## 1. Quick Configuration, API Key, Tunnel Provider, Keep-Alive & GPU Check


In [ ]:
import glob
import json
import os
import re
import secrets
import shutil
import site
import socket
import subprocess
import tarfile
import time
import urllib.request

# Auto-detect execution platform: 'kaggle', 'colab', or 'local'
if os.path.exists("/kaggle/working"):
    PLATFORM = "kaggle"
elif "COLAB_RELEASE_TAG" in os.environ or os.path.exists("/content"):
    PLATFORM = "colab"
else:
    PLATFORM = "local"

# ==========================================
# 1. MODEL & SERVER CONFIGURATION
# ==========================================
REPO_ID = "BoldingBuilds/Ternary-Bonsai-2-27B-Abliterated-v2-PQ2_0-MTP-GGUF"
PRISM_RELEASE_TAG = "prism-b10828-a153697"
USE_MTP = True               # True: use MTP variant + speculative drafting; False if image inputs error
USE_VISION = True            # True: enable image input (loads from Dataset/Drive or downloads 0.63 GB)
IMAGE_MIN_TOKENS = 1024      # Higher value improves small text/detail recognition in images
CONTEXT_SIZE = 131072        # 128K context (lower to 32768 on a single 16 GB Colab T4 if OOM occurs)
KV_CACHE_TYPE = "q8_0"       # 8-bit quantized KV cache to save GPU VRAM
DEFAULT_THINKING = False     # False: disable thinking by default; True: enable thinking by default
REASONING_EFFORT = "medium"  # Used when DEFAULT_THINKING = True ("low", "medium", or "high")
SERVER_PORT = 8081

# Google Colab option: store/load the bundle on Google Drive to avoid re-downloading
COLAB_USE_GOOGLE_DRIVE = False
COLAB_DRIVE_BUNDLE_DIR = "/content/drive/MyDrive/bonsai_bundle"

# ==========================================
# 2. API KEY AUTHENTICATION & PUBLIC TUNNEL SETTINGS
# ==========================================
REQUIRE_API_KEY = True          # True: enforce Bearer API key auth so knowing the Base URL alone is not enough
API_KEY = ""                    # Optional manual key; recommended to store 'API_KEY' in Kaggle/Colab Secrets instead.
                                # If left empty and not in Secrets (with REQUIRE_API_KEY=True), a random key is auto-generated.
SHOW_API_KEY_IN_OUTPUT = False  # False: mask keys loaded from Secrets in cell logs; auto-generated keys are always shown once

# Choose public tunnel provider:
#   "auto"       -> Uses Ngrok if NGROK_AUTHTOKEN is set; otherwise automatically falls back to Cloudflare Quick Tunnel (*.trycloudflare.com)
#   "ngrok"      -> Forces Ngrok (requires NGROK_AUTHTOKEN; supports your static NGROK_DOMAIN)
#   "cloudflare" -> Forces Cloudflare Quick Tunnel (free random *.trycloudflare.com domain, no account or secrets needed)
TUNNEL_PROVIDER = "auto"

MANUAL_NGROK_AUTHTOKEN = ""     # Optional fallback if 'NGROK_AUTHTOKEN' is not set in Secrets (leave blank to use Cloudflare in 'auto' mode)
MANUAL_NGROK_DOMAIN = ""        # Optional static Ngrok domain fallback if not set in Secrets (used with Ngrok only)

# ==========================================
# 3. AUTO KEEP-ALIVE CONFIGURATION
# ==========================================
ENABLE_KEEP_ALIVE = True             # True: run continuous keep-alive monitoring loop; False: exit Cell 3 after startup
KEEP_ALIVE_MIN_HOURS = 5.0           # Minimum runtime to maintain (hours), e.g. 5.0, 6.0, 8.0, 11.5
KEEP_ALIVE_PING_INTERVAL = 45        # Health-check & heartbeat interval (seconds) — recommended 30-60s
KEEP_ALIVE_GPU_PULSE_SEC = 180       # Interval (seconds) for tiny 1-token GPU pulse to prevent GPU-idle termination
KEEP_ALIVE_AUTO_RESTART = True       # Automatically restart llama-server or the active tunnel if either stops unexpectedly
KEEP_ALIVE_ENABLE_BROWSER_JS = True  # Inject JS to auto-dismiss 'Are you still there?' / reconnect popups

os.environ["PATH"] = "/opt/bin:/usr/local/cuda/bin:" + os.environ.get("PATH", "")
smi = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    text=True,
).strip()
print(f"Detected Platform : {PLATFORM.upper()}")
print(f"GPU(s)            : {smi.replace(chr(10), ' | ')}")
print(f"Tunnel Provider   : {TUNNEL_PROVIDER.upper()} (Ngrok if token set, else free Cloudflare Quick Tunnel)")
print(f"API Key Auth      : {'ENABLED (Required)' if REQUIRE_API_KEY else 'DISABLED (Public — Not Recommended)'}")
print(
    f"Keep-Alive        : {'ENABLED' if ENABLE_KEEP_ALIVE else 'DISABLED'} "
    f"| Target {KEEP_ALIVE_MIN_HOURS}h | Ping every {KEEP_ALIVE_PING_INTERVAL}s"
)


## 2. Install CUDA 12 Runtime + Tunnel Dependencies & Load Binary/Model/Vision Assets


In [ ]:
!pip install -q pyngrok nvidia-cuda-runtime-cu12 nvidia-cublas-cu12 hf_transfer huggingface_hub

import requests
from huggingface_hub import hf_hub_download
from pyngrok import conf, ngrok
from pyngrok.installer import install_ngrok

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

if PLATFORM == "colab" and COLAB_USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(COLAB_DRIVE_BUNDLE_DIR, exist_ok=True)

SEARCH_ROOTS = []
if PLATFORM == "kaggle":
    SEARCH_ROOTS.extend(["/kaggle/input", "/kaggle/working"])
elif PLATFORM == "colab":
    if COLAB_USE_GOOGLE_DRIVE:
        SEARCH_ROOTS.append(COLAB_DRIVE_BUNDLE_DIR)
    SEARCH_ROOTS.extend(["/content/bonsai_bundle", "/content"])
else:
    SEARCH_ROOTS.append(os.getcwd())

LOCAL_FALLBACK_DIR = (
    COLAB_DRIVE_BUNDLE_DIR
    if (PLATFORM == "colab" and COLAB_USE_GOOGLE_DRIVE)
    else ("/content/bonsai_bundle" if PLATFORM == "colab" else "/tmp/bonsai_bundle")
)
os.makedirs(LOCAL_FALLBACK_DIR, exist_ok=True)


def find_existing_file(filename):
    """Search known local/Dataset/Drive roots for an already-downloaded file."""
    for root_dir in SEARCH_ROOTS:
        if os.path.exists(root_dir):
            matches = glob.glob(os.path.join(root_dir, "**", filename), recursive=True)
            if matches:
                return matches[0]
    return None


def get_cuda_lib_paths(bin_dir):
    """Build LD_LIBRARY_PATH including binary directory and pip-installed NVIDIA CUDA 12 libraries."""
    paths = [bin_dir, "/usr/local/nvidia/lib64", "/usr/local/cuda/lib64"]
    for sp in site.getsitepackages():
        paths.extend(glob.glob(os.path.join(sp, "nvidia", "*", "lib")))
    return ":".join(p for p in dict.fromkeys(paths) if os.path.exists(p))


def test_binary(bin_path):
    """Verify that llama-server executes cleanly and detects CUDA devices."""
    env = os.environ.copy()
    env["LD_LIBRARY_PATH"] = get_cuda_lib_paths(os.path.dirname(bin_path)) + ":" + env.get("LD_LIBRARY_PATH", "")
    res = subprocess.run([bin_path, "--list-devices"], env=env, capture_output=True, text=True)
    return res.returncode == 0, (res.stdout + "\n" + res.stderr).strip()


# 1. Load llama-server binary from Dataset/Drive or download fresh
TMP_BIN_DIR = "/tmp/prism-bin"
LLAMA_SERVER_BIN = None

existing_server = find_existing_file("llama-server")
if existing_server:
    SRC_BIN_DIR = os.path.dirname(existing_server)
    if os.path.exists(TMP_BIN_DIR):
        shutil.rmtree(TMP_BIN_DIR)
    shutil.copytree(SRC_BIN_DIR, TMP_BIN_DIR, symlinks=True)
    for root, _, files in os.walk(TMP_BIN_DIR):
        for fn in files:
            fp = os.path.join(root, fn)
            if not os.path.islink(fp):
                os.chmod(fp, 0o755)
            m = re.match(r"^(lib.+\.so\.0)\..+$", fn)
            if m:
                link_path = os.path.join(root, m.group(1))
                if not os.path.exists(link_path):
                    os.symlink(fn, link_path)
    cand = os.path.join(TMP_BIN_DIR, "llama-server")
    ok, out = test_binary(cand)
    if ok:
        LLAMA_SERVER_BIN = cand
        print(f"Loaded llama-server from existing storage ({SRC_BIN_DIR}).")
    else:
        print(f"Existing binary is missing libraries ({out[:160]}). Downloading fresh package (~3s)...")

if not LLAMA_SERVER_BIN:
    if os.path.exists(TMP_BIN_DIR):
        shutil.rmtree(TMP_BIN_DIR)
    os.makedirs(TMP_BIN_DIR, exist_ok=True)
    url = (
        f"https://github.com/PrismML-Eng/llama.cpp/releases/download/"
        f"{PRISM_RELEASE_TAG}/llama-{PRISM_RELEASE_TAG}-bin-linux-cuda-12.8-x64.tar.gz"
    )
    tar_path = "/tmp/prism-cuda128.tar.gz"
    urllib.request.urlretrieve(url, tar_path)
    with tarfile.open(tar_path, "r:gz") as tf:
        tf.extractall(TMP_BIN_DIR)
    os.remove(tar_path)
    found = glob.glob(os.path.join(TMP_BIN_DIR, "**", "llama-server"), recursive=True)
    if not found:
        raise RuntimeError("llama-server binary not found after extracting archive.")
    LLAMA_SERVER_BIN = found[0]
    os.chmod(LLAMA_SERVER_BIN, 0o755)
    ok, out = test_binary(LLAMA_SERVER_BIN)
    if not ok:
        raise RuntimeError(f"llama-server failed verification: {out}")
    print("Downloaded and verified llama-server binary successfully!")

# 2. Locate or download the GGUF model file
MODEL_FILENAME = (
    "Ternary-Bonsai-2-27B-Abliterated-v2-PQ2_0-MTP.gguf"
    if USE_MTP
    else "Ternary-Bonsai-2-27B-Abliterated-v2-PQ2_0.gguf"
)
MODEL_PATH = find_existing_file(MODEL_FILENAME)
if MODEL_PATH:
    print("Model (from Dataset/Drive):", MODEL_PATH)
else:
    if PLATFORM == "kaggle":
        print(f"No attached Dataset contains {MODEL_FILENAME}. Downloading directly to {LOCAL_FALLBACK_DIR}...")
    else:
        print(f"Downloading {MODEL_FILENAME} to {LOCAL_FALLBACK_DIR}...")
    MODEL_PATH = hf_hub_download(repo_id=REPO_ID, filename=MODEL_FILENAME, local_dir=LOCAL_FALLBACK_DIR)
    print("Model (downloaded):", MODEL_PATH)

# 3. Locate or download the Vision mmproj file (if USE_VISION = True)
MMPROJ_PATH = None
if USE_VISION:
    MMPROJ_FILENAME = "Ternary-Bonsai-2-27B-mmproj-Q8_0.gguf"
    MMPROJ_PATH = find_existing_file(MMPROJ_FILENAME)
    if MMPROJ_PATH:
        print("Vision mmproj (from Dataset/Drive):", MMPROJ_PATH)
    else:
        print("Downloading Vision mmproj (0.63 GB)...")
        MMPROJ_PATH = hf_hub_download(repo_id=REPO_ID, filename=MMPROJ_FILENAME, local_dir=LOCAL_FALLBACK_DIR)
        print("Vision mmproj (downloaded):", MMPROJ_PATH)


## 3. Launch `llama-server` (with API Key Auth) + Ngrok / Cloudflare Tunnel + Auto Keep-Alive Loop


In [ ]:
import datetime
import sys


def is_port_free(port):
    """Return True if localhost TCP port is available."""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("127.0.0.1", port)) != 0


def get_secret(name, fallback=""):
    """Read a secret from environment variables, Kaggle/Colab Secrets, or a manual fallback."""
    val = os.environ.get(name, "").strip()
    if val:
        return val
    if PLATFORM == "kaggle":
        try:
            from kaggle_secrets import UserSecretsClient
            val = UserSecretsClient().get_secret(name)
            if val:
                return val.strip()
        except Exception:
            pass
    elif PLATFORM == "colab":
        try:
            from google.colab import userdata
            val = userdata.get(name)
            if val:
                return val.strip()
        except Exception:
            pass
    return fallback.strip()


def mask_secret(val):
    """Return a masked display string for sensitive keys."""
    if len(val) <= 10:
        return "*" * len(val)
    return f"{val[:6]}...{val[-4:]}"


def start_llama_server(port, api_key=""):
    """Start llama-server on localhost with optional Bearer API key enforcement."""
    subprocess.run(["pkill", "-f", "llama-server"], check=False)
    time.sleep(1)
    while not is_port_free(port):
        port += 1

    if DEFAULT_THINKING:
        chat_kwargs = json.dumps({"reasoning_effort": REASONING_EFFORT})
        thinking_flags = []
    else:
        chat_kwargs = json.dumps({"enable_thinking": False})
        thinking_flags = ["--reasoning-budget", "0"]

    cmd = [
        LLAMA_SERVER_BIN,
        "-m", MODEL_PATH,
        "-ngl", "99",
        "-fa", "on",
        "-c", str(CONTEXT_SIZE),
        "--parallel", "1",
        "--cache-type-k", KV_CACHE_TYPE,
        "--cache-type-v", KV_CACHE_TYPE,
        "--jinja",
        "--chat-template-kwargs", chat_kwargs,
        "--temp", "1.0",
        "--top-k", "20",
        "--top-p", "0.95",
        "--metrics",
        "--host", "127.0.0.1",
        "--port", str(port),
    ] + thinking_flags

    # Enforce API key authentication on all /v1/* endpoints
    if api_key:
        cmd.extend(["--api-key", api_key])
    if USE_MTP:
        cmd.extend(["--spec-type", "draft-mtp", "--spec-draft-n-max", "2"])
    if MMPROJ_PATH:
        cmd.extend(["--mmproj", MMPROJ_PATH, "--image-min-tokens", str(IMAGE_MIN_TOKENS)])

    env = os.environ.copy()
    env["LD_LIBRARY_PATH"] = get_cuda_lib_paths(os.path.dirname(LLAMA_SERVER_BIN)) + ":" + env.get("LD_LIBRARY_PATH", "")

    log_path = "/tmp/llama_server.log"
    proc = subprocess.Popen(cmd, stdout=open(log_path, "w"), stderr=subprocess.STDOUT, env=env)

    print(f"Loading model into GPU VRAM on port {port}...")
    ready = False
    for _ in range(120):
        if proc.poll() is not None:
            with open(log_path, "r") as f:
                print(f.read()[-3000:])
            raise RuntimeError(f"llama-server exited early with code {proc.returncode}")
        try:
            # /health stays public even when --api-key is set
            if requests.get(f"http://127.0.0.1:{port}/health", timeout=2).status_code == 200:
                ready = True
                break
        except Exception:
            pass
        time.sleep(2)

    if not ready:
        raise TimeoutError("llama-server did not respond to /health in time.")
    return proc, port


def verify_api_key_protection(port, api_key):
    """Confirm that unauthenticated /v1/models requests are blocked (401) and valid keys succeed (200)."""
    if not api_key:
        return
    try:
        no_auth = requests.get(f"http://127.0.0.1:{port}/v1/models", timeout=5)
        with_auth = requests.get(
            f"http://127.0.0.1:{port}/v1/models",
            headers={"Authorization": f"Bearer {api_key}"},
            timeout=5,
        )
        if no_auth.status_code == 401 and with_auth.status_code == 200:
            print("[SECURITY] Verified: unauthenticated /v1/models blocked (HTTP 401); valid API key accepted (HTTP 200).")
        else:
            print(
                f"[SECURITY] Warning: unexpected auth check status "
                f"(no-key={no_auth.status_code}, with-key={with_auth.status_code})."
            )
    except Exception as e:
        print(f"[SECURITY] Could not complete auth self-check: {e}")


def start_ngrok_tunnel(port, authtoken, domain):
    """Launch an Ngrok HTTPS tunnel forwarding public traffic to localhost:<port>."""
    try:
        ngrok.kill()
    except Exception:
        pass
    subprocess.run(["pkill", "-f", "ngrok"], check=False)
    time.sleep(1)

    ngrok_path = conf.get_default().ngrok_path
    if not os.path.exists(ngrok_path):
        install_ngrok(ngrok_path)
    subprocess.run([ngrok_path, "config", "add-authtoken", authtoken], check=True)

    ngrok_cmd = [ngrok_path, "http", str(port), "--log=stdout"]
    if domain:
        ngrok_cmd.extend(["--domain", domain])
    proc = subprocess.Popen(ngrok_cmd, stdout=open("/tmp/ngrok.log", "w"), stderr=subprocess.STDOUT)

    public_url = None
    for _ in range(30):
        try:
            tunnels = requests.get("http://127.0.0.1:4040/api/tunnels", timeout=2).json().get("tunnels", [])
            https_tunnels = [t for t in tunnels if t.get("public_url", "").startswith("https://")]
            if https_tunnels:
                public_url = https_tunnels[0]["public_url"].rstrip("/")
                break
        except Exception:
            pass
        time.sleep(1)

    if not public_url:
        raise TimeoutError("Failed to retrieve PUBLIC_URL from Ngrok.")
    return proc, public_url


def start_cloudflare_tunnel(port):
    """Download cloudflared (if needed) and launch a free zero-config Cloudflare Quick Tunnel (*.trycloudflare.com)."""
    subprocess.run(["pkill", "-f", "cloudflared"], check=False)
    time.sleep(1)

    cf_bin = shutil.which("cloudflared") or "/tmp/cloudflared"
    if not os.path.exists(cf_bin):
        print("[TUNNEL] Downloading cloudflared binary for Linux x64...")
        cf_url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
        urllib.request.urlretrieve(cf_url, cf_bin)
        os.chmod(cf_bin, 0o755)

    cf_log = "/tmp/cloudflared.log"
    cf_cmd = [cf_bin, "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{port}"]
    proc = subprocess.Popen(cf_cmd, stdout=open(cf_log, "w"), stderr=subprocess.STDOUT)

    public_url = None
    url_pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")
    for _ in range(40):
        if proc.poll() is not None:
            with open(cf_log, "r") as f:
                print(f.read()[-2000:])
            raise RuntimeError(f"cloudflared exited early with code {proc.returncode}")
        try:
            with open(cf_log, "r") as f:
                m = url_pattern.search(f.read())
                if m:
                    public_url = m.group(0).rstrip("/")
                    break
        except Exception:
            pass
        time.sleep(1)

    if not public_url:
        raise TimeoutError("Failed to retrieve *.trycloudflare.com URL from cloudflared.")

    # Brief pause so the new trycloudflare.com DNS record finishes propagating
    time.sleep(3)
    return proc, public_url


def start_public_tunnel(provider, port, authtoken, domain):
    """Start either Ngrok or Cloudflare Quick Tunnel based on TUNNEL_PROVIDER ('auto', 'ngrok', or 'cloudflare')."""
    mode = (provider or "auto").strip().lower()
    if mode not in {"auto", "ngrok", "cloudflare"}:
        raise ValueError(f"Invalid TUNNEL_PROVIDER='{provider}'. Use 'auto', 'ngrok', or 'cloudflare'.")

    if mode == "auto":
        if authtoken:
            active = "ngrok"
        else:
            active = "cloudflare"
            print("[TUNNEL] No NGROK_AUTHTOKEN found — automatically using a free Cloudflare Quick Tunnel (*.trycloudflare.com).")
    else:
        active = mode

    if active == "ngrok":
        if not authtoken:
            raise ValueError(
                "TUNNEL_PROVIDER='ngrok' requires NGROK_AUTHTOKEN! "
                "Add it to Secrets, set MANUAL_NGROK_AUTHTOKEN, or switch TUNNEL_PROVIDER to 'auto' / 'cloudflare'."
            )
        proc, url = start_ngrok_tunnel(port, authtoken, domain)
        return "ngrok", proc, url

    proc, url = start_cloudflare_tunnel(port)
    return "cloudflare", proc, url


def is_tunnel_alive(active_provider, proc):
    """Check whether the active tunnel process (Ngrok or Cloudflare) is still up."""
    if proc is None or proc.poll() is not None:
        return False
    if active_provider == "ngrok":
        try:
            tunnels = requests.get("http://127.0.0.1:4040/api/tunnels", timeout=3).json().get("tunnels", [])
            return len(tunnels) > 0
        except Exception:
            return False
    return True


def inject_browser_keepalive():
    """Inject JavaScript to prevent browser tab sleep/idle disconnects on Colab and Kaggle."""
    try:
        from IPython.display import Javascript, display
        js_code = """
        (function() {
            if (window.__keepAliveInjected) return;
            window.__keepAliveInjected = true;
            setInterval(function() {
                // 1. Auto-click Colab reconnect / confirmation dialogs
                const colabBtn = document.querySelector('colab-connect-button') || document.querySelector('#top-toolbar colab-connect-button');
                if (colabBtn && colabBtn.shadowRoot) {
                    const inner = colabBtn.shadowRoot.querySelector('#connect');
                    if (inner && inner.innerText && inner.innerText.toLowerCase().includes('reconnect')) {
                        inner.click();
                    }
                }
                document.querySelectorAll('mwc-button, paper-button, button').forEach(b => {
                    const txt = (b.innerText || b.textContent || '').trim().toLowerCase();
                    if (txt === 'ok' || txt === 'reconnect' || txt === 'keep working' || txt === 'stay connected') {
                        b.click();
                    }
                });
                // 2. Dispatch light mouse activity to keep the tab active
                document.dispatchEvent(new MouseEvent('mousemove', {bubbles: true, clientX: Math.random()*100, clientY: Math.random()*100}));
            }, 30000);
            console.log('[Keep-Alive] Browser anti-idle script active.');
        })();
        """
        display(Javascript(js_code))
    except Exception:
        pass


def run_gpu_pulse(port, api_key=""):
    """Send a tiny 1-token authenticated request so Kaggle/Colab does not flag the GPU as idle."""
    try:
        headers = {"Authorization": f"Bearer {api_key}"} if api_key else {}
        payload = {
            "model": MODEL_FILENAME,
            "messages": [{"role": "user", "content": "ping"}],
            "max_tokens": 1,
            "temperature": 0.0,
        }
        r = requests.post(
            f"http://127.0.0.1:{port}/v1/chat/completions",
            headers=headers,
            json=payload,
            timeout=15,
        )
        return r.status_code == 200
    except Exception:
        return False


# Resolve optional Ngrok credentials (if empty & TUNNEL_PROVIDER='auto', Cloudflare Quick Tunnel is used automatically)
NGROK_AUTHTOKEN = get_secret("NGROK_AUTHTOKEN", MANUAL_NGROK_AUTHTOKEN)
NGROK_DOMAIN = re.sub(r"^https?://", "", get_secret("NGROK_DOMAIN", MANUAL_NGROK_DOMAIN)).strip("/")

# Resolve API Key (Secrets 'API_KEY' -> Cell 1 API_KEY -> auto-generated secure key)
ACTIVE_API_KEY = ""
API_KEY_SOURCE = "DISABLED"
if REQUIRE_API_KEY:
    ACTIVE_API_KEY = get_secret("API_KEY", API_KEY)
    if ACTIVE_API_KEY:
        API_KEY_SOURCE = "Secrets/Config"
    else:
        ACTIVE_API_KEY = f"sk-bonsai-{secrets.token_hex(24)}"
        API_KEY_SOURCE = "Auto-Generated"

server_proc, ACTIVE_PORT = start_llama_server(SERVER_PORT, ACTIVE_API_KEY)
verify_api_key_protection(ACTIVE_PORT, ACTIVE_API_KEY)
ACTIVE_TUNNEL_PROVIDER, tunnel_proc, PUBLIC_URL = start_public_tunnel(
    TUNNEL_PROVIDER, ACTIVE_PORT, NGROK_AUTHTOKEN, NGROK_DOMAIN
)

if not ACTIVE_API_KEY:
    api_key_display = "DISABLED (WARNING: anyone with your Base URL can use this server!)"
elif API_KEY_SOURCE == "Auto-Generated" or SHOW_API_KEY_IN_OUTPUT:
    api_key_display = f"{ACTIVE_API_KEY} ({API_KEY_SOURCE})"
else:
    api_key_display = f"{mask_secret(ACTIVE_API_KEY)} ({API_KEY_SOURCE} — set SHOW_API_KEY_IN_OUTPUT=True to reveal)"

tunnel_label = (
    "NGROK (Static/Account)" if ACTIVE_TUNNEL_PROVIDER == "ngrok" else "CLOUDFLARE QUICK TUNNEL (Random Free Domain)"
)

print("=" * 74)
print(f"PLATFORM         : {PLATFORM.upper()}")
print(f"TUNNEL PROVIDER  : {tunnel_label}")
print(f"OPENAI BASE URL  : {PUBLIC_URL}/v1")
print(f"MODEL ID         : {MODEL_FILENAME}")
print(f"API KEY          : {api_key_display}")
print(f"CONTEXT / CACHE  : {CONTEXT_SIZE} ({KV_CACHE_TYPE}) | Vision={bool(MMPROJ_PATH)}")
print(f"DEFAULT THINKING : {'ON' if DEFAULT_THINKING else 'OFF'}")
print("=" * 74)

# ==========================================
# AUTO KEEP-ALIVE MONITORING LOOP
# ==========================================
if ENABLE_KEEP_ALIVE:
    if KEEP_ALIVE_ENABLE_BROWSER_JS:
        inject_browser_keepalive()

    target_seconds = int(KEEP_ALIVE_MIN_HOURS * 3600)
    start_ts = time.time()
    last_gpu_pulse_ts = start_ts
    end_dt = datetime.datetime.now() + datetime.timedelta(seconds=target_seconds)

    print(
        f"\n[KEEP-ALIVE] Maintaining session for at least {KEEP_ALIVE_MIN_HOURS} hours "
        f"(estimated end: {end_dt.strftime('%H:%M:%S %d/%m')})."
    )
    print("[KEEP-ALIVE] Click the Stop button (■) on this cell anytime to stop monitoring manually.\n")

    try:
        while True:
            now = time.time()
            elapsed = now - start_ts
            if elapsed >= target_seconds:
                print(f"\n[KEEP-ALIVE] Reached target uptime of {KEEP_ALIVE_MIN_HOURS} hours!")
                break

            # 1. Check llama-server health
            server_alive = (server_proc.poll() is None)
            if server_alive:
                try:
                    r = requests.get(f"http://127.0.0.1:{ACTIVE_PORT}/health", timeout=5)
                    server_alive = (r.status_code == 200)
                except Exception:
                    server_alive = False

            if not server_alive:
                print(f"\n[{datetime.datetime.now().strftime('%H:%M:%S')}] Warning: llama-server stopped responding!")
                if KEEP_ALIVE_AUTO_RESTART:
                    print("[KEEP-ALIVE] Restarting llama-server with API key protection...")
                    server_proc, ACTIVE_PORT = start_llama_server(ACTIVE_PORT, ACTIVE_API_KEY)
                else:
                    break

            # 2. Check active tunnel (Ngrok or Cloudflare)
            if not is_tunnel_alive(ACTIVE_TUNNEL_PROVIDER, tunnel_proc):
                print(f"\n[{datetime.datetime.now().strftime('%H:%M:%S')}] Warning: {ACTIVE_TUNNEL_PROVIDER.upper()} tunnel disconnected!")
                if KEEP_ALIVE_AUTO_RESTART:
                    print(f"[KEEP-ALIVE] Reopening {ACTIVE_TUNNEL_PROVIDER.upper()} tunnel...")
                    ACTIVE_TUNNEL_PROVIDER, tunnel_proc, PUBLIC_URL = start_public_tunnel(
                        ACTIVE_TUNNEL_PROVIDER, ACTIVE_PORT, NGROK_AUTHTOKEN, NGROK_DOMAIN
                    )
                    print(f"[KEEP-ALIVE] New Base URL: {PUBLIC_URL}/v1")
                else:
                    break

            # 3. Periodic lightweight GPU pulse (authenticated with ACTIVE_API_KEY)
            gpu_pulsed = False
            if (now - last_gpu_pulse_ts) >= KEEP_ALIVE_GPU_PULSE_SEC:
                gpu_pulsed = run_gpu_pulse(ACTIVE_PORT, ACTIVE_API_KEY)
                last_gpu_pulse_ts = now

            # 4. Print rolling status heartbeat to keep stdout active
            hrs_done = elapsed / 3600.0
            hrs_left = max(0.0, (target_seconds - elapsed) / 3600.0)
            status_line = (
                f"\r[{datetime.datetime.now().strftime('%H:%M:%S')}] "
                f"Uptime: {hrs_done:.2f}h / {KEEP_ALIVE_MIN_HOURS:.1f}h ({hrs_left:.2f}h left) | "
                f"Server: OK | Tunnel ({ACTIVE_TUNNEL_PROVIDER.upper()}): OK"
                + (" | GPU Pulse: OK" if gpu_pulsed else "")
            )
            sys.stdout.write(status_line + "   ")
            sys.stdout.flush()

            time.sleep(KEEP_ALIVE_PING_INTERVAL)
    except KeyboardInterrupt:
        print("\n[KEEP-ALIVE] Monitoring loop stopped by user (llama-server & tunnel remain running in the background).")


## 4. Clean Shutdown for Server & Tunnels (Does Not Auto-Run on `Run All`)


In [ ]:
def stop_all():
    """Terminate Ngrok, Cloudflare Tunnel (cloudflared), and llama-server cleanly."""
    try:
        ngrok.kill()
    except Exception:
        pass
    subprocess.run(["pkill", "-f", "ngrok"], check=False)
    subprocess.run(["pkill", "-f", "cloudflared"], check=False)
    subprocess.run(["pkill", "-f", "llama-server"], check=False)
    print("Stopped Ngrok, Cloudflare Tunnel, and llama-server cleanly.")

# Uncomment the line below when you want to shut down the server manually:
# stop_all()
